## Downscaling and moving windows
- Downscaling DTM rasters to 30m to have same-level scale with LST images 
- computing moving windows and study the correlations with LST depending on the size of the window.

### Libraries and general paths

In [ ]:
!pip install rasterio

In [ ]:
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter
from rasterio import mask
import geopandas as gpd



In [ ]:
lst_cropped_path = '../../datos-notebooks/0.LST/LST_cropped_celsius.tif' # values out of bound are 0
lst_clipped_path = '../../datos-notebooks/0.LST/LST_clipped_celsius.tif' #values out of bound are nan
dtm_path = '../../datos-notebooks/3.topography/dtm_mosaic.tif'

output_dir = '../../datos-notebooks/3.topography/dtm_30m/'
os.makedirs(output_dir, exist_ok=True)

bound = "../../datos-notebooks/bologne_boundary.geojson"
gdf = gpd.read_file(bound)


### Downscaling to 30m aligned with LST raster

In [ ]:
#print a window of the dtm

with rasterio.open(dtm_path) as src:
    window = rasterio.windows.Window(col_off=15000, row_off=13000, width=8000, height=8000)
    data = src.read(1, window=window)
    transform = src.window_transform(window)

# --- Plot ---
fig, ax = plt.subplots(figsize=(10,10))
img = ax.imshow(data, cmap='YlGn', vmin=0, vmax=250,)
ax.set_title("DTM mosaic (sample window)")
plt.colorbar(img, ax=ax, label="terrain height")
plt.show()


#### Downscale to 30m

In [ ]:
# Downscaling

# Save raster function
def save_raster(filename, array):
    profile = lst_profile.copy()
    profile.update(dtype=rasterio.float32, count=1, compress='lzw')
    with rasterio.open(os.path.join(output_dir, filename), 'w', **profile) as dst:
        dst.write(array.astype(np.float32), 1)

# Read LST cropped to get reference profile
with rasterio.open(lst_cropped_path) as lst_ref:
    lst_profile = lst_ref.profile
    lst_transform = lst_ref.transform
    lst_crs = lst_ref.crs
    lst_shape = (lst_ref.height, lst_ref.width)

with rasterio.open(dtm_path) as dtm_src:
    dtm_data = dtm_src.read(1)
    dtm_profile = dtm_src.profile

    # Force CRS if missing
    if dtm_src.crs is None:
        from rasterio.crs import CRS
        dtm_crs = CRS.from_epsg(7791)  # EPSG:7791 
    else:
        dtm_crs = dtm_src.crs

    dtm_resampled = np.empty(lst_shape, dtype=np.float32)

    reproject(
        source=dtm_data,
        destination=dtm_resampled,
        src_transform=dtm_src.transform,
        src_crs=dtm_crs,            # use the CRS we defined
        dst_transform=lst_transform,
        dst_crs=lst_crs,
        resampling=Resampling.average
    )

save_raster('dtm_30m.tif', dtm_resampled)

#### extract:
- mean elevation
- mean slope
- northness
- eastness
- aspect

In [ ]:
# Extend borders before computing gradients
dtm_pad = np.pad(dtm_resampled, pad_width=1, mode='edge') # Extend array by one pixel in each direction, repeating edge values

# Pixel size (xres, yres)
xres = lst_transform.a
yres = -lst_transform.e  

# Compute gradients on padded array
dz_dy, dz_dx = np.gradient(dtm_pad, yres, xres)

# Remove padding to restore original shape
dz_dy = dz_dy[1:-1, 1:-1]
dz_dx = dz_dx[1:-1, 1:-1]

# Compute slope and aspect
# Slope in radians
slope = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
# Aspect (0 = North)
aspect = np.arctan2(dz_dx, -dz_dy)
aspect = np.where(aspect < 0, 2 * np.pi + aspect, aspect)

# Compute northness and eastness
northness = np.cos(aspect)
eastness = np.sin(aspect)

save_raster('elev_mean.tif', dtm_resampled)
save_raster('slope_mean.tif', np.degrees(slope))  # degrees
save_raster('northness.tif', northness)
save_raster('eastness.tif', eastness)
save_raster('aspect.tif', np.degrees(aspect)) #in degrees, where 0 is nord

print("Finished. Files saved as elev_mean.tif, slope_mean.tif, northness.tif, eastness.tif, aspect.tif")

In [ ]:
# plot
files = {
    "Mean elevation (m)": os.path.join(output_dir, "elev_mean.tif"),
    "Mean slope (°)": os.path.join(output_dir, "slope_mean.tif"),
    "Northness": os.path.join(output_dir, "northness.tif"),
    "Eastness": os.path.join(output_dir, "eastness.tif"), 
    "Aspect (°)": os.path.join(output_dir, "aspect.tif")
}


fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for ax, (title, path) in zip(axes, files.items()):
    with rasterio.open(path) as src:
        data = src.read(1)
        data = np.where(np.isfinite(data), data, np.nan)  # mask invalid values

    cmap = 'terrain' if 'Elevación' in title else 'RdYlBu_r'

    if 'elevation' in title:
        im = ax.imshow(data, cmap=cmap, vmin=0, vmax=500)  
    else:
        im = ax.imshow(data, cmap=cmap)

    ax.set_title(title, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


### Moving windows

In [ ]:
files = {
    "elev_mean": os.path.join(output_dir, "elev_mean.tif"),
    "slope_mean": os.path.join(output_dir, "slope_mean.tif"),
    "northness": os.path.join(output_dir, "northness.tif"),
    "eastness": os.path.join(output_dir, "eastness.tif"),
    "aspect": os.path.join(output_dir, "aspect.tif"),
}


# Prepare geometry in CRS of first raster
with rasterio.open(list(files.values())[0]) as ref:
    boundary_crs = gdf.to_crs(ref.crs)
geoms = [boundary_crs.geometry.union_all().__geo_interface__]


window_sizes = range(1, 301)

for var_name, path in files.items():
    var_dir = os.path.join(output_dir, var_name)
    os.makedirs(var_dir, exist_ok=True)

    with rasterio.open(path) as src:
        data = src.read(1)
        profile = src.profile.copy()

    for w in window_sizes:
        # Moving window mean
        avg_window = uniform_filter(data, size=w, mode='reflect')

        # Clip to city boundary
        with rasterio.MemoryFile() as memfile:
            with memfile.open(driver='GTiff', 
                              height=avg_window.shape[0],
                              width=avg_window.shape[1],
                              count=1,
                              dtype='float32',
                              transform=profile['transform'],
                              nodata=np.nan) as tmp:
                tmp.write(avg_window.astype(np.float32), 1)
                clipped_arr, clipped_transform = mask.mask(tmp, geoms, crop=True, nodata=np.nan)

        clipped_arr = clipped_arr.squeeze().astype(np.float32)

        # Update profile
        profile.update({
            "height": clipped_arr.shape[0],
            "width": clipped_arr.shape[1],
            "transform": clipped_transform,
            "dtype": "float32",
            "nodata": np.nan
        })

        # Save raster
        out_name = f"{var_name}_win{w}.tif"
        out_path = os.path.join(var_dir, out_name)
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(clipped_arr, 1)

print("Finished. All variables clipped and saved.")


In [ ]:
# Compute relative elevation (micro-relief)
rel_dir = os.path.join(output_dir, "rel_elev")
os.makedirs(rel_dir, exist_ok=True)

with rasterio.open(files['elev_mean']) as src:
    elev = src.read(1)
    profile = src.profile

for w in range(1, 301):
    elev_mean_window = moving_window_mean(elev, w)
    rel_elev = elev - elev_mean_window
    out_name = f"rel_elev_win{w}.tif"
    out_path = os.path.join(rel_dir, out_name)
    profile.update(dtype=rasterio.float32, compress='lzw')
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(rel_elev.astype(np.float32), 1)



In [ ]:
# Clip LST raster 

with rasterio.open(lst_cropped_path) as src:
    # Convert boundary to raster CRS
    boundary_crs = gdf.to_crs(src.crs)
    geoms = [boundary_crs.geometry.union_all().__geo_interface__]  # avoids deprecation warning

    # Clip raster to city boundary
    clipped_arr, clipped_transform = mask.mask(
        src,
        geoms,
        crop=True,
        nodata=np.nan
    )

    # Convert to float and handle original nodata
    clipped_arr = clipped_arr.astype("float32")
    if src.nodata is not None:
        clipped_arr[clipped_arr == src.nodata] = np.nan

    # Remove extra dimensions
    clipped_arr = clipped_arr.squeeze()

    # Update metadata
    meta = src.meta.copy()
    meta.update({
        "height": clipped_arr.shape[0],
        "width": clipped_arr.shape[1],
        "transform": clipped_transform,
        "dtype": "float32",
        "nodata": np.nan
    })

# Overwrite original LST raster
with rasterio.open(lst_clipped_path, "w", **meta) as dst:
    dst.write(clipped_arr, 1)

print("LST raster clipped and overwritten successfully.")


In [ ]:
# Plots
window_sizes = range(1, 11)  # 1 a 10
variables = ["elev_mean", "slope_mean", "northness", "eastness", "rel_elev", "aspect"]

for var_name in variables:
    var_folder = os.path.join(output_dir, var_name)

    # Determine vmin and vmax for Elevation and Rel Elev
    if var_name == 'elev_mean':
        vmin, vmax = 0, 500  
    elif var_name == 'rel_elev':
        vmin, vmax = -50, 50  
    else:
        vmin, vmax = None, None  # auto scale for the rest

    # Create subplot grid
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()

    for idx, w in enumerate(window_sizes):
        raster_path = os.path.join(var_folder, f"{var_name}_win{w}.tif")
        with rasterio.open(raster_path) as src_var:
            var_data = src_var.read(1).astype(float)

        # Plot
        im = axes[idx].imshow(var_data, cmap='terrain' if 'elev' in var_name else 'RdYlBu_r',
                              vmin=vmin, vmax=vmax)
        axes[idx].set_title(f"Win {w}")
        axes[idx].axis('off')
        plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)

    plt.suptitle(f"{var_name} - Windows 1 to 10", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


### correlations with LST

In [ ]:
with rasterio.open(lst_clipped_path) as lst:
    lst_crs = lst.crs
    lst_clip = lst.read(1)          
    lst_transform = lst.transform 

In [ ]:
window_sizes = range(1, 51)
variables = ["elev_mean", "slope_mean", "northness", "eastness", "rel_elev", "aspect"]

for var_name in variables:
    var_folder = os.path.join(output_dir, var_name)
    correlations = []

    for w in window_sizes:
        raster_path = os.path.join(var_folder, f"{var_name}_win{w}.tif")
        with rasterio.open(raster_path) as src_var:
            var_data = src_var.read(1).astype(float)  # read full raster

        # Pixel-by-pixel correlation with LST
        valid_mask = (~np.isnan(lst_clip))  # only LST valid pixels
        lst_vals = lst_clip[valid_mask]
        var_vals = var_data[valid_mask]

        corr = np.corrcoef(var_vals, lst_vals)[0,1]
        correlations.append(corr)

    # Plot correlation vs window size
    plt.figure(figsize=(8,5))
    plt.plot(window_sizes, correlations, marker='o')
    plt.xlabel("Size of window (pixels)")
    plt.ylabel(f"Correlation {var_name} – LST")
    plt.title(f"Correlation {var_name} vs window size")
    plt.grid(True)
    plt.show()


In [ ]:
# Print more windows to see more context
window_sizes = range(1, 151)

variables = ["elev_mean", "slope_mean", "northness", "eastness", "rel_elev", "aspect"]

for var_name in variables:
    var_folder = os.path.join(output_dir, var_name)
    correlations = []

    for w in window_sizes:
        raster_path = os.path.join(var_folder, f"{var_name}_win{w}.tif")
        with rasterio.open(raster_path) as src_var:
            var_data = src_var.read(1).astype(float)  # read full raster

        # Pixel-by-pixel correlation with LST
        valid_mask = (~np.isnan(lst_clip))  # only LST valid pixels
        lst_vals = lst_clip[valid_mask]
        var_vals = var_data[valid_mask]

        corr = np.corrcoef(var_vals, lst_vals)[0,1]
        correlations.append(corr)

    #Plot
    plt.figure(figsize=(8,5))
    plt.plot(window_sizes, correlations, marker='o')
    plt.xlabel("Window size (pixels)")
    plt.ylabel(f"Correlation {var_name} – LST")
    plt.title(f"Correlation {var_name} vs window size")
    plt.grid(True)
    plt.show()
